IMPORTANT

If you haven't already, create a .venv and install dependencies

After creating a .venv, run this in your code editor's terminal inside the .venv source location: "python -m pip install pandas ipykernel numpy jupyter"

In [1]:
import pandas as pd
import numpy as np

In [10]:
play_attention_scores = pd.read_csv("../outputs_csv/play_attention_scores.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")


**VERY IMPORTANT!**

The files below are not in the GitHub because they are too large. They are in a .zip file in the shared Google Drive folder inside the "Output Datasets" folder. The zip file is called "Datasets Not in Github (Michael).zip". 

After extracting the .zip file, drag the base_filtered.csv file and the pass_rushers.csv file into the outputs_csv folder.

In [3]:
# The base_filtered dataframe contains frame-by-frame data of all players on only relevant plays.
base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")

# The pass_rushers dataframe contains data of only pass rushers in the pass rush window.
pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")

/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_18451/2742888671.py:2: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")
/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_18451/2742888671.py:5: DtypeWarning: Columns (0: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")


In [11]:
pffScoutingData

,gameId,playId,nflId,pff_role,pff_positionLinedUp,pff_nflIdBlockedPlayer,pff_blockType,pff_backFieldBlock
0,2021090900,97,25511,Pass,QB,NaN,NaN,NaN
1,2021090900,97,35481,Pass Route,TE-L,NaN,NaN,NaN
2,2021090900,97,35634,Pass Route,LWR,NaN,NaN,NaN
3,2021090900,97,39985,Pass Route,HB-R,NaN,NaN,NaN
4,2021090900,97,40151,Pass Block,C,44955.0,SW,0.0
...,...,...,...,...,...,...,...,...
188249,2021110100,4433,52507,Pass Block,LT,43338.0,PP,0.0
188250,2021110100,4433,52546,Coverage,SCBoR,NaN,NaN,NaN
188251,2021110100,4433,52573,Pass Route,SLoWR,NaN,NaN,NaN
188252,2021110100,4433,52585,Pass Rush,LEO,NaN,NaN,NaN


,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,x,y,s,a,dis,o,dir,event,frameIdEndWindow,pff_positionLinedUp_y
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36,QB
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.200,...,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN,36,QB
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.300,...,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN,36,QB
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.400,...,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN,36,QB
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.500,...,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN,36,QB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5166915,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,40.36,44.25,7.47,0.28,0.74,48.35,60.83,NaN,37,LWR
5166916,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,41.02,44.60,7.45,0.67,0.75,48.35,61.74,NaN,37,LWR
5166917,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,41.68,44.94,7.48,1.43,0.75,58.99,63.79,NaN,37,LWR
5166918,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,42.36,45.25,7.41,2.01,0.75,63.66,65.69,NaN,37,LWR


In [5]:
# ball_snap_frames contains only frames when the ball is snapped
ball_snap_frames = base_filtered[base_filtered['event'] == 'ball_snap']

# each_pass_rusher contains one row for each pass rusher on each play.
each_pass_rusher = pass_rushers[["gameId", "playId", "frameId", "nflId", "displayName"]].drop_duplicates(
    subset=['gameId','playId','nflId'])

# base_filtered_pass_rushers is for pass rushers only, and it includes frames before the pass rush window unlike pass_rushers.
base_filtered_pass_rushers = base_filtered.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')

Feature-engineer variables for how far the pass rusher is from the center at the ball_snap instance (x distance, y distance, and total distance).

In [ ]:
# filter ball_snap_frames to only include ball snap frames of pass rushers in each_pass_rusher df
ball_snaps_pass_rushers = ball_snap_frames.merge(each_pass_rusher[['gameId', 'playId', 'nflId']].drop_duplicates(), on=['gameId', 'playId', 'nflId'], how='inner')
ball_snaps_pass_rushers

# ball_snap_frames to only include centers
ball_snaps_centers = ball_snap_frames[ball_snap_frames['pff_positionLinedUp'] == 'C']
ball_snaps_centers

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,43.30,18.89,0.96,0.90,0.11,316.48,288.76,ball_snap,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,43.90,32.63,0.37,2.44,0.06,278.77,247.75,ball_snap,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,43.35,25.18,0.56,2.42,0.06,243.27,288.42,ball_snap,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,43.68,21.93,0.36,2.85,0.02,277.16,316.78,ball_snap,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,right,43.70,26.67,0.03,0.18,0.01,31.86,345.84,ball_snap,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,right,30.29,28.64,0.13,0.39,0.04,286.55,249.67,ball_snap,32
30967,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,right,30.31,29.79,0.11,0.12,0.06,250.03,271.87,ball_snap,37
30968,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,right,30.19,20.82,0.22,1.21,0.03,250.01,251.87,ball_snap,37
30969,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,right,30.45,26.65,0.81,2.68,0.07,261.06,283.80,ball_snap,37


In [ ]:
# calculate the difference in x, the difference in y, and the euclidean distance of each pass rusher to the center at the moment of the ball snap on each play.
temp = ball_snaps_pass_rushers.merge(ball_snaps_centers[['gameId', 'playId', 'x', 'y']], on=['gameId', 'playId'], how='inner', suffixes=('', '_center'))
temp['diff_x'] = temp['x'] - temp['x_center']
temp['diff_y'] = temp['y'] - temp['y_center']
temp['euclidean_distance'] = np.sqrt(temp['diff_x']**2 + temp['diff_y']**2)
temp

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,dis,o,dir,event,frameIdEndWindow,x_center,y_center,diff_x,diff_y,euclidean_distance
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.11,316.48,288.76,ball_snap,36,42.10,24.02,1.20,-5.13,5.268482
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.06,278.77,247.75,ball_snap,36,42.10,24.02,1.80,8.61,8.796141
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.06,243.27,288.42,ball_snap,36,42.10,24.02,1.25,1.16,1.705315
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.02,277.16,316.78,ball_snap,36,42.10,24.02,1.58,-2.09,2.620019
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.600,...,0.01,31.86,345.84,ball_snap,36,42.10,24.02,1.60,2.65,3.095561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30966,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:40.200,...,0.04,286.55,249.67,ball_snap,32,29.07,23.78,1.22,4.86,5.010788
30967,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.06,250.03,271.87,ball_snap,37,29.15,23.72,1.16,6.07,6.179846
30968,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.03,250.01,251.87,ball_snap,37,29.15,23.72,1.04,-2.90,3.080844
30969,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:21.800,...,0.07,261.06,283.80,ball_snap,37,29.15,23.72,1.30,2.93,3.205448


In [ ]:
# Run a multiple regression model. The three independent variables are diff_x, diff_y, and euclidean_distance in temp. 
# The dependent variable is avg_attention_score in play_attention_scores. 

# first merge temp with play_attention_scores on gameId, playId, and nflId to get the avg_attention_score for each pass rusher at the moment of the ball snap.
temp = temp.merge(play_attention_scores.rename(columns={'rusher_nflId': 'nflId'}), on=['gameId', 'playId', 'nflId'], how='inner')
X = temp[['diff_x', 'diff_y', 'euclidean_distance']]
y = temp['avg_attention_score']
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X, y)
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

# r2score
from sklearn.metrics import r2_score
y_pred = model.predict(X)
print("R^2 Score:", r2_score(y, y_pred))


Coefficients: [ 0.00154197  0.00039535 -0.14094692]
Intercept: 1.7347556649748586
R^2 Score: 0.2990456656993984


In [28]:
# Calculate gravity for Myles Garrett based on this multiple regression model.
myles_garrett_temp = temp[temp['displayName'] == 'Myles Garrett']
myles_garrett_temp['predicted_attention_score'] = model.predict(myles_garrett_temp[['diff_x', 'diff_y', 'euclidean_distance']])
myles_garrett_temp['gravity_score'] = myles_garrett_temp['avg_attention_score'] - myles_garrett_temp['predicted_attention_score']
myles_garrett_avg_gravity = myles_garrett_temp['gravity_score'].mean()
print("Myles Garrett's average gravity:", myles_garrett_avg_gravity)  

# Calculate gravity for Aaron Donald based on this multiple regression model.
aaron_donald_temp = temp[temp['displayName'] == 'Aaron Donald']
aaron_donald_temp['predicted_attention_score'] = model.predict(aaron_donald_temp[['diff_x', 'diff_y', 'euclidean_distance']])
aaron_donald_temp['gravity_score'] = aaron_donald_temp['avg_attention_score'] - aaron_donald_temp['predicted_attention_score']
aaron_donald_avg_gravity = aaron_donald_temp['gravity_score'].mean()
print("Aaron Donald's average gravity:", aaron_donald_avg_gravity)  

Myles Garrett's average gravity: 0.15320782400613006
Aaron Donald's average gravity: 0.3504502102932687
